# ESM2 Masked Language Model

This notebook demonstrates the three core operations of the ESM2 tool: extracting sequence embeddings, sampling mutations, and scoring sequences. ESM2 is a protein language model trained on millions of protein sequences using masked language modeling, and its learned representations capture structural and evolutionary relationships that support a range of downstream tasks.

In [19]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("esm2")
display_overview("esm2")
display_docs_section("esm2", "Background")

# ESM2

> ESM2 (Evolutionary Scale Modeling 2) is Meta AI's [protein language model](https://www.evolutionaryscale.ai/blog/esm-cambrian) trained on millions of protein sequences from [UniRef](https://www.uniprot.org/help/uniref). It provides sequence embeddings, per-position amino acid logits, sequence mutation (sampling), and sequence scoring (MLM pseudo-perplexity). ESM2 offers multiple model sizes from 8M to 15B parameters, balancing quality and computational cost.

## Background

**What are protein language models?**
Protein language models (pLMs) learn the "grammar" of proteins from evolutionary data. They capture:

* **Sequence conservation**: Which residues are essential for function
* **[Co-evolution](https://en.wikipedia.org/wiki/Coevolution)**: Pairs of residues that evolve together (often in contact)
* **Structural constraints**: Patterns that define [secondary/tertiary structure](https://en.wikipedia.org/wiki/Protein_structure)
* **Functional motifs**: Binding sites, active sites, [post-translational modifications](https://en.wikipedia.org/wiki/Post-translational_modification)

**Why embeddings are useful:**
ESM2 embeddings encode rich biological information:

* Similar proteins cluster in embedding space
* Embeddings predict structure, function, and localization
* Mean-pooled embeddings work as fixed-length sequence descriptors

**Why logits are useful:**
Per-position logits indicate the model's confidence in each amino acid:

* High logits = evolutionarily preferred (conserved)
* Low logits = tolerated or deleterious positions
* Comparing wild-type vs mutant logits predicts variant effects
* Logits are returned over 20 canonical amino acids in fixed order: `ACDEFGHIKLMNPQRSTVWY`

## Available tools

In [20]:
display_available_tools("esm2")

- **`run_esm2_embeddings()`** — Extract protein sequence embeddings and logits using ESM2
- **`run_esm2_sample()`** — Sample protein sequences using ESM2 language model
- **`run_esm2_score()`** — Score protein sequences using ESM2 language model

### `run_esm2_embeddings`

ESM2 produces dense vector representations for each input protein sequence. These mean-pooled embeddings compress the full sequence-length representation into a single fixed-dimension vector that captures structural, functional, and evolutionary properties. Embeddings from different sequences can be compared directly using cosine similarity or Euclidean distance, making them useful as features for machine learning models, for clustering related proteins, or for building sequence search indices.

In [21]:
from proto_tools import (
    ESM2EmbeddingsConfig,
    ESM2EmbeddingsInput,
    run_esm2_embeddings,
)

In [22]:
# Display docs
display_api_reference("esm2", "input", "run_esm2_embeddings")

# Two hemoglobin-like sequences to embed
sequences = [
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
    "MNLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
]
inputs = ESM2EmbeddingsInput(sequences=sequences)

**Input** — `MaskedModelInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Protein sequence(s) to process. Can be provided as: |

In [23]:
# Display config docs
display_api_reference("esm2", "config", "run_esm2_embeddings")

# Configure the 650M model | see docs above for all fields
config = ESM2EmbeddingsConfig(
    model_checkpoint="esm2_t33_650M_UR50D",
    batch_size=2,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESM2EmbeddingsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_checkpoint` | `enum` | `esm2_t33_650M_UR50D` | Name of the ESM2 model variant to use. Options: Available options: `esm2_t6_8M_UR50D`, `esm2_t12_35M_UR50D`, `esm2_t30_150M_UR50D`, `esm2_t33_650M_UR50D`, `esm2_t36_3B_UR50D`, `esm2_t48_15B_UR50D` |
| `return_logits` | `boolean` | `False` | Whether to include per-position logits in the output. When `True`, returns logits for each sequence. When `False`, only returns metrics (saves memory and serialization time). Default: `False`. |
| `batch_size` | `integer` | `1` | Number of sequences to process in parallel. Larger batches improve throughput but require more GPU memory. Optimal values depend on GPU memory, model size, and sequence lengths. Typical values range from 1 (safest) to 128 (faster, more memory). Default: 1. |
| `verbose` | `boolean` | `False` | Whether to print status messages during model execution, including loading progress and timing information. Default: `False`. |
| `device` | `string` | `cuda` | Device to run the model on. Options include `"cuda"` (NVIDIA GPU), `"cpu"` (CPU execution), `"mps"` (Apple Metal), or specific GPU devices like `"cuda:0"`. Default: `"cuda"`. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [24]:
# Run the embeddings function
embeddings_result = run_esm2_embeddings(inputs, config)

Running run_esm2_embeddings [00:00]

In [25]:
# Display output docs
display_api_reference("esm2", "output", "run_esm2_embeddings")

# Inspect the embedding dimension and first few values for each sequence
for i, emb in enumerate(embeddings_result.results):
    print(f"Sequence {i + 1}: embedding length = {len(emb.mean_embedding)}, first 5 values = {emb.mean_embedding[:5]}")

**Output** — `ESM2EmbeddingsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `List[SequenceEmbedding]` | required | Per-sequence embedding results. Each `SequenceEmbedding` contains: |
| `mean_embedding` | `List[number]` | required | Mean-pooled embedding vector for one sequence. |
| `attention_mask` | `List[integer]` | required | Binary mask indicating valid positions (1) vs padding (0). |
| `logits` | `array` |  | Optional per-position amino acid logits for one sequence. |

Sequence 1: embedding length = 1280, first 5 values = [-0.010943022556602955, -0.07043583691120148, -0.04228722304105759, 0.03445248305797577, -0.08040354400873184]
Sequence 2: embedding length = 1280, first 5 values = [-0.013179367408156395, -0.06401950865983963, -0.04112844541668892, 0.034877192229032516, -0.07414667308330536]


### `run_esm2_sample`

ESM2 sampling uses the model's predicted amino acid distributions to propose mutations at masked positions. There are two ways to specify which positions to sample:

1. **Custom masks** — directly mark positions with `_` in the input sequence to control exactly which residues are redesigned
2. **Masking strategy** — let the `MaskingStrategy` config automatically select positions using random, entropy-based, or max-logit methods

#### Approach 1: Custom masks with `_`

Use the `_` character to explicitly mark which positions should be redesigned. All other positions are held fixed. This is useful when you know exactly which residues you want to mutate — for example, redesigning a loop region or a specific binding interface.

In [26]:
from proto_tools import (
    ESM2SampleConfig,
    ESM2SampleInput,
    run_esm2_sample,
)

# Display input docs
display_api_reference("esm2", "input", "run_esm2_sample")

# Mask positions 6-10 with _ to redesign that region
masked_input = ESM2SampleInput(sequences=["MVLSP_____VKAAW"])

**Input** — `MaskedModelInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Protein sequence(s) to process. Can be provided as: |

In [27]:
# Display config docs
display_api_reference("esm2", "config", "run_esm2_sample")

# No masking strategy needed — positions are already specified by _
config = ESM2SampleConfig(
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESM2SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `masking_strategy` | `MaskingStrategy` |  | Controls which positions to mask for sampling. Default: random 30%. |
| `method` | `enum` | `random` | Scoring method for position selection. `"random"`: uniform random, `"entropy"`: highest model uncertainty, `"max-logit"`: lowest model confidence. Available options: `random`, `entropy`, `max-logit` |
| `num_mutations` | `integer` |  | Exact number of positions to mask per sequence. |
| `mask_fraction` | `number` |  | Fraction of designable positions to mask (e.g. 0.15 for \~15%). |
| `fixed_positions` | `array` |  | 1-indexed positions that must NOT be masked. Applied uniformly to all sequences. |
| `temperature` | `number` | `1.0` | Temperature for position selection. \ 1.0 is more uniform. |
| `model_name` | `string` |  | Which masked model to use for scoring. Defaults to the sampling tool's model when unset. |
| `model_checkpoint` | `enum` | `esm2_t33_650M_UR50D` | ESM2 model variant. Default: `"esm2_t33_650M_UR50D"`. Available options: `esm2_t6_8M_UR50D`, `esm2_t12_35M_UR50D`, `esm2_t30_150M_UR50D`, `esm2_t33_650M_UR50D`, `esm2_t36_3B_UR50D`, `esm2_t48_15B_UR50D` |
| `temperature` | `number` | `1.0` | Sampling temperature (\ 1.0 diverse). Default: 1.0. |
| `batch_size` | `integer` | `1` | Sequences per GPU forward pass. Default: 1. |
| `return_logits` | `boolean` | `False` | Whether to include per-position logits. Default: `False`. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run on. Default: `"cuda"`. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [28]:
# Run sampling on the custom-masked input
custom_mask_result = run_esm2_sample(masked_input, config)

Running run_esm2_sample [00:00]

In [29]:
# Display output docs
display_api_reference("esm2", "output", "run_esm2_sample")

# Show the original vs sampled — only the masked positions change
original = "MVLSPADKTNVKAAW"
for i, seq in enumerate(custom_mask_result.sequences):
    diffs = [j + 1 for j, (a, b) in enumerate(zip(original, seq)) if a != b]
    print(f"Original: {original}")
    print(f"Sampled:  {seq}  (mutated positions: {diffs})")

**Output** — `ESM2SampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Sampled or mutated protein sequences. Each sequence is a string of amino acid characters. For de novo generation, these are completely new sequences. For mutation, these are modified versions of the input sequences with specified positions changed to model-predicted alternatives. |
| `logits` | `array` |  | Per-position logits for each sequence. Shape is (num\_sequences, seq\_len, vocab\_size=20). Only present if return\_logits=True in config. |

Original: MVLSPADKTNVKAAW
Sampled:  MVLSPKYGWRVKAAW  (mutated positions: [6, 7, 8, 9, 10])


#### Approach 2: Automatic position selection with `MaskingStrategy`

Instead of manually placing `_` characters, you can let `MaskingStrategy` choose which positions to mask. The `method` parameter controls the selection logic:

- **`"random"`** (default) — uniform random selection
- **`"entropy"`** — targets positions where ESM2 is least certain about the original residue
- **`"max-logit"`** — targets positions where ESM2 most confidently predicts a different amino acid

Use `num_mutations` for an exact count or `mask_fraction` for a proportion of positions.

In [30]:
from proto_tools.tools.masked_models.masking import MaskingStrategy

# Entropy-based masking to target the 5 most uncertain positions
strategy_input = ESM2SampleInput(sequences=["MVLSPADKTNVKAAW"])
strategy_config = ESM2SampleConfig(
    masking_strategy=MaskingStrategy(method="entropy", num_mutations=5),
    device="cuda",  # Change to "cpu" if no GPU available
)

strategy_result = run_esm2_sample(strategy_input, strategy_config)

Running run_esm2_embeddings [00:00]

Running run_esm2_sample [00:00]

In [31]:
# The model automatically selected the 5 highest-entropy positions to mutate
original = "MVLSPADKTNVKAAW"
for i, seq in enumerate(strategy_result.sequences):
    diffs = [j + 1 for j, (a, b) in enumerate(zip(original, seq)) if a != b]
    print(f"Original: {original}")
    print(f"Sampled:  {seq}  (mutated positions: {diffs})")

Original: MVLSPADKTNVKAAW
Sampled:  MVLSPADKAGQKGAM  (mutated positions: [9, 10, 11, 13, 15])


### `run_esm2_score`

Pseudo-perplexity scoring masks each position in a sequence one at a time and measures how well the model predicts the original residue. Lower perplexity indicates that the sequence is more consistent with the evolutionary patterns the model learned during training, while higher perplexity suggests the sequence contains unusual or unlikely amino acid choices. This is useful for ranking designed sequences, filtering out poor candidates before expensive experimental validation, or identifying anomalous regions within a protein.

In [32]:
from proto_tools import (
    ESM2ScoringConfig,
    ESM2ScoringInput,
    run_esm2_score,
)

In [33]:
# Display docs
display_api_reference("esm2", "input", "run_esm2_score")

# Two short sequences to score
inputs = ESM2ScoringInput(sequences=["MKTAYIAKQR", "EVQLVESGGS"])

**Input** — `MaskedModelInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Protein sequence(s) to process. Can be provided as: |

In [34]:
# Display config docs
display_api_reference("esm2", "config", "run_esm2_score")

# Process 5 masked variants per forward pass | see docs above for all fields
config = ESM2ScoringConfig(
    batch_size=5,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESM2ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_checkpoint` | `enum` | `esm2_t33_650M_UR50D` | ESM2 model checkpoint to use. Options include `"esm2_t6_8M_UR50D"` through `"esm2_t48_15B_UR50D"`. Available options: `esm2_t6_8M_UR50D`, `esm2_t12_35M_UR50D`, `esm2_t30_150M_UR50D`, `esm2_t33_650M_UR50D`, `esm2_t36_3B_UR50D`, `esm2_t48_15B_UR50D` |
| `batch_size` | `integer` | `1` | Number of masked sequence variants to process per forward pass. For a sequence of length L, scoring requires L forward passes (one per position). This parameter controls how many of those masked variants are batched together. Larger batches improve throughput but use more GPU memory; reduce if encountering out-of-memory errors. |
| `return_logits` | `boolean` | `False` | Whether to include per-position logits in the output. When `True`, returns logits for each sequence. When `False`, only returns metrics (saves memory and serialization time). Default: `False`. |
| `verbose` | `boolean` | `False` | Whether to print status messages during scoring. |
| `device` | `string` | `cuda` | Device to run the model on. Options include `"cuda"`, `"cpu"`, `"mps"`, or specific GPU devices like `"cuda:0"`. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [35]:
# Run the scoring function
score_result = run_esm2_score(inputs, config)

Running run_esm2_score [00:00]

In [36]:
# Display output docs
display_api_reference("esm2", "output", "run_esm2_score")

# Show all scoring metrics for each input sequence
for i, score in enumerate(score_result.scores):
    print(f"Sequence {i + 1}: {inputs.sequences[i]}")
    print(f"  Log-likelihood:     {score.metrics['log_likelihood']:.3f}")
    print(f"  Avg log-likelihood: {score.metrics['avg_log_likelihood']:.3f}")
    print(f"  Perplexity:         {score.metrics['perplexity']:.3f}")

**Output** — `MaskedModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `List[SequenceScores]` | required | List of scoring outputs, one per input sequence. Each entry contains metrics (log\_likelihood, avg\_log\_likelihood, perplexity) and optional per-position logits. |
| `metrics` | `Dict[string, number]` |  | Dictionary of scalar scoring metrics. |
| `logits` | `array` |  | Optional per-position logits array. |
| `vocab` | `array` |  | Optional token ordering for logits; logits\[:, j] corresponds to vocab\[j]. |

Sequence 1: MKTAYIAKQR
  Log-likelihood:     -27.243
  Avg log-likelihood: -2.724
  Perplexity:         15.246
Sequence 2: EVQLVESGGS
  Log-likelihood:     -28.396
  Avg log-likelihood: -2.840
  Perplexity:         17.110


## Export Results

Results from each function can be exported to standard file formats for downstream analysis. Embeddings export to CSV with one row per sequence, sampled sequences export to FASTA for compatibility with other bioinformatics tools, and scores export to JSON with full metadata.

In [37]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

embeddings_result.export("esm2_embeddings", export_path=output_dir, file_format="csv")
print(f"Embeddings exported to {output_dir / 'esm2_embeddings.csv'}")

strategy_result.export("esm2_sequences", export_path=output_dir, file_format="fasta")
print(f"Sampled sequences exported to {output_dir / 'esm2_sequences.fasta'}")

score_result.export("esm2_scores", export_path=output_dir, file_format="json")
print(f"Scores exported to {output_dir / 'esm2_scores.json'}")

Embeddings exported to example_output/esm2_embeddings.csv
Sampled sequences exported to example_output/esm2_sequences.fasta
Scores exported to example_output/esm2_scores.json
